<a href="https://colab.research.google.com/github/Rakshitanaik-27/EliteTech-DataScience/blob/main/Task1_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# EliteTech Intern - Task 1: Data Pipeline Development (ETL)
# Purpose: Build an automated ETL pipeline using Pandas and Scikit-Learn
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ------------------------------------------------------------------------------
# STEP 1: EXTRACT (Load Raw Data)
# ------------------------------------------------------------------------------
print("--- STEP 1: EXTRACTING RAW DATA ---")
# Load the raw Titanic dataset directly from a public URL
dataset_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
raw_df = pd.read_csv(dataset_url)

print(f"Raw dataset loaded successfully! Total Rows: {raw_df.shape[0]}, Columns: {raw_df.shape[1]}")

# ------------------------------------------------------------------------------
# STEP 2: TRANSFORM (Data Preprocessing & Feature Engineering Pipeline)
# ------------------------------------------------------------------------------
print("\n--- STEP 2: TRANSFORMING DATA ---")

# Select relevant features for transformation
features = raw_df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()

numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# 1. Pipeline for Numerical Features:
#    - Impute missing values (NaN) with Median
#    - Scale values using StandardScaler (converts to Z-scores)
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Pipeline for Categorical Features:
#    - Impute missing values (NaN) with Most Frequent value (Mode)
#    - Convert text categories into binary columns using OneHotEncoder
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Combine both numerical and categorical pipelines using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, numeric_features),
    ('cat', cat_pipeline, categorical_features)
])

# Execute pipeline transformation on selected features
transformed_data = preprocessor.fit_transform(features)

# Retrieve generated column names from OneHotEncoder
cat_encoder = preprocessor.named_transformers_['cat']['encoder']
encoded_cat_cols = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_column_names = numeric_features + encoded_cat_cols

# Create clean DataFrame
clean_df = pd.DataFrame(transformed_data, columns=all_column_names)
clean_df['Survived'] = raw_df['Survived']  # Append target column back

print("Data transformation completed successfully!")

# ------------------------------------------------------------------------------
# STEP 3: LOAD (Save Cleaned Data)
# ------------------------------------------------------------------------------
print("\n--- STEP 3: LOADING CLEAN DATA ---")

# Save the transformed and cleaned dataset to a new CSV file
output_filename = "cleaned_titanic_data.csv"
clean_df.to_csv(output_filename, index=False)

print(f"Cleaned dataset saved as '{output_filename}'")

# Preview output sample
print("\n--- PROCESSED DATA SAMPLE ---")
print(clean_df.head())

--- STEP 1: EXTRACTING RAW DATA ---
Raw dataset loaded successfully! Total Rows: 891, Columns: 12

--- STEP 2: TRANSFORMING DATA ---
Data transformation completed successfully!

--- STEP 3: LOADING CLEAN DATA ---
Cleaned dataset saved as 'cleaned_titanic_data.csv'

--- PROCESSED DATA SAMPLE ---
        Age      Fare     SibSp     Parch  Pclass_1  Pclass_2  Pclass_3  \
0 -0.565736 -0.502445  0.432793 -0.473674       0.0       0.0       1.0   
1  0.663861  0.786845  0.432793 -0.473674       1.0       0.0       0.0   
2 -0.258337 -0.488854 -0.474545 -0.473674       0.0       0.0       1.0   
3  0.433312  0.420730  0.432793 -0.473674       1.0       0.0       0.0   
4  0.433312 -0.486337 -0.474545 -0.473674       0.0       0.0       1.0   

   Sex_female  Sex_male  Embarked_C  Embarked_Q  Embarked_S  Survived  
0         0.0       1.0         0.0         0.0         1.0         0  
1         1.0       0.0         1.0         0.0         0.0         1  
2         1.0       0.0         0.0  